# Stage B6 — Visual Demo: One Index, Two Tiers, Live

An interactive walkthrough of the validated architecture:

1. **Server gallery** — photos indexed server-side (native SigLIP), displayed and labeled
2. **Mobile upload** — you upload a photo; it is stored in Drive ("mobile demo storage") and indexed through the *phone path* (MobileCLIP -> fp16 adapter)
3. **Auto-caption** — a generated sentence describing your photo
4. **Search prompt** — you type a query for a server photo or your mobile photo
5. **Top-3 results with probabilities** — thumbnails, tier labels, softmax scores
6. **Timing** — the same shared index answers both tiers; local (mobile) search skips the network and is faster
7. **Flow log** — every step of upload / search / fetch / display / describe is printed as it happens

Prerequisites in `DATA_DIR`: `adapter.npz` (B2) and the `coco/` cache (B1).
Runtime: GPU recommended (works on CPU, slower model loads).


In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
# 2. Load models + adapter + flow logger
import numpy as np, torch, json, time, random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

DATA_DIR = Path(os.environ['DATA_DIR'])
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def flow(tier, *steps):
    """Requirement 7: print the flow of every operation."""
    stamp = time.strftime('%H:%M:%S')
    print(f'[FLOW {stamp}][{tier}] ' + '  ->  '.join(steps))

import open_clip
mob, _, mob_pre = open_clip.create_model_and_transforms(
    'MobileCLIP-S1', pretrained='datacompdr')
mob.eval().to(DEV)
mob_tok = open_clip.get_tokenizer('MobileCLIP-S1')
flow('setup', 'MobileCLIP-S1 loaded (phone encoder, 30MB-class)')

from transformers import AutoModel, AutoProcessor
SIG = 'google/siglip2-base-patch16-224'
sig = AutoModel.from_pretrained(SIG).eval().to(DEV)
sig_proc = AutoProcessor.from_pretrained(SIG)
flow('setup', 'SigLIP 2 loaded (server encoder, 1.5GB-class)')

# BLIP loaded directly (the 'image-to-text' pipeline task was removed
# in newer transformers versions)
from transformers import BlipProcessor, BlipForConditionalGeneration
blip_proc = BlipProcessor.from_pretrained(
    'Salesforce/blip-image-captioning-base')
blip = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base').eval().to(DEV)

@torch.no_grad()
def caption_image(img):
    x = blip_proc(images=img, return_tensors='pt').to(DEV)
    out = blip.generate(**x, max_new_tokens=30)
    return blip_proc.decode(out[0], skip_special_tokens=True).strip()

flow('setup', 'BLIP captioner loaded (for photo descriptions)')

W = np.load(DATA_DIR / 'adapter.npz')['W_ridge']
W16 = W.astype(np.float16).astype(np.float32)
flow('setup', f'adapter loaded {W.shape} (the 0.79MB fp16 matrix)')

# ---- the SHARED INDEX: one list, entries tagged by tier ----
INDEX = []   # dicts: {vec (768,), tier, path, name}

def l2(v):
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-9)

@torch.no_grad()
def siglip_embed_image(img):                     # server-side indexing
    x = sig_proc(images=[img], return_tensors='pt').to(DEV)
    e = sig.get_image_features(**x)
    e = e if torch.is_tensor(e) else e.pooler_output
    return l2(e.float().cpu().numpy())[0]

@torch.no_grad()
def phone_embed_image(img):                      # mobile-side indexing
    x = mob_pre(img).unsqueeze(0).to(DEV)
    e = torch.nn.functional.normalize(mob.encode_image(x), dim=-1)
    return l2(e.float().cpu().numpy() @ W16)[0]  # adapter, verbatim math

@torch.no_grad()
def siglip_embed_text(q):                        # query encoder
    t = sig_proc(text=[q], return_tensors='pt', padding='max_length',
                 truncation=True, max_length=64).to(DEV)
    e = sig.get_text_features(**t)
    e = e if torch.is_tensor(e) else e.pooler_output
    return l2(e.float().cpu().numpy())[0]

## Step 1 — The SERVER gallery
The six *smallest* COCO photos (quick to display) are indexed **server-side
with native SigLIP** and shown below, clearly labeled as living on the
server.

In [ ]:
# 3. Build + display the server gallery (smallest images by file size)
img_dir = DATA_DIR / 'coco' / 'val2017'
ann = json.load(open(DATA_DIR / 'coco' / 'annotations' /
                     'captions_val2017.json'))
id2file = {im['id']: im['file_name'] for im in ann['images']}
id2cap = {}
for a in ann['annotations']:
    id2cap.setdefault(a['image_id'], a['caption'])

sized = sorted(((img_dir / f).stat().st_size, i, f)
               for i, f in id2file.items() if i in id2cap
               and (img_dir / f).exists())
server_picks = sized[:6]                    # requirement 1: smallest images

fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for ax, (sz, i, f) in zip(axes.flat, server_picks):
    img = Image.open(img_dir / f).convert('RGB')
    vec = siglip_embed_image(img)
    INDEX.append(dict(vec=vec, tier='SERVER', path=img_dir / f,
                      name=id2cap[i][:45]))
    flow('server', f'index {f} ({sz//1024}KB)', 'SigLIP encode 768d',
         'write to SHARED index [tier=SERVER]')
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'SERVER | {id2cap[i][:34]}', fontsize=8, color='#1a5276')
plt.suptitle('These 6 photos live ON THE SERVER '
             '(indexed with native SigLIP)', fontsize=12)
plt.tight_layout(); plt.show()
print(f'\nShared index now holds {len(INDEX)} server vectors.')

## Step 2 + 3 — Upload YOUR photo to the mobile tier, get a description
The photo is saved to Drive under `mobile_demo_storage/` (playing the role
of the phone's storage), indexed through the **phone path**
(MobileCLIP -> fp16 adapter), and captioned automatically.

In [ ]:
# 4. Upload (requirement 2), index via phone path, caption (requirement 3)
from google.colab import files
MOBILE_DIR = DATA_DIR / 'mobile_demo_storage'
MOBILE_DIR.mkdir(exist_ok=True)

print('Choose a photo to upload to the MOBILE tier...')
up = files.upload()
for fname, data in up.items():
    dest = MOBILE_DIR / fname
    dest.write_bytes(data)
    flow('mobile', f'photo received ({len(data)//1024}KB)',
         f'saved to Drive: mobile_demo_storage/{fname}')

    img = Image.open(dest).convert('RGB')
    t0 = time.perf_counter()
    vec = phone_embed_image(img)
    t_enc = (time.perf_counter() - t0) * 1000
    flow('mobile', f'MobileCLIP encode 512d + adapter fp16 -> 768d '
         f'({t_enc:.0f}ms)', 'write to SHARED index [tier=MOBILE]')

    cap = caption_image(img)
    flow('mobile', 'BLIP caption generated')
    INDEX.append(dict(vec=vec, tier='MOBILE', path=dest, name=cap[:60]))

    plt.figure(figsize=(4.6, 4.6))
    plt.imshow(img); plt.axis('off')
    plt.title(f'MOBILE | "{cap}"', fontsize=9, color='#0f766e')
    plt.show()
    print(f'\nGenerated description: "{cap}"')
print(f'Shared index now holds {len(INDEX)} vectors '
      f'({sum(e["tier"]=="SERVER" for e in INDEX)} server, '
      f'{sum(e["tier"]=="MOBILE" for e in INDEX)} mobile).')

## Step 4 + 5 + 6 — Search the ONE shared index; top-3 with probabilities; timing
Type a query about a **server photo** or **your mobile photo**. The same
768-d index answers both. Timing compares the *local (mobile) route* —
pure in-memory search — against the *server round-trip route* (identical
search + a simulated 150ms cellular network RTT, marked as simulated).

In [ ]:
# 5. Interactive search (requirements 4, 5, 6, 7)
NETWORK_RTT_MS = 150.0   # simulated phone->server->phone cellular round trip

def search_and_show(query, k=3, temperature=0.02):
    flow('search', f'query text: "{query}"', 'SigLIP text encode 768d')
    q = siglip_embed_text(query)

    mat = np.stack([e['vec'] for e in INDEX])

    t0 = time.perf_counter()
    sims = mat @ q                                   # the shared index
    order = np.argsort(sims)[::-1][:k]
    t_local = (time.perf_counter() - t0) * 1000
    t_server = t_local + NETWORK_RTT_MS

    flow('search@MOBILE', f'local in-memory search over shared index '
         f'({len(INDEX)} vecs)', f'{t_local:.2f}ms total')
    flow('search@SERVER', 'same shared index', 'same math',
         f'+{NETWORK_RTT_MS:.0f}ms simulated network RTT',
         f'{t_server:.2f}ms total')

    # probabilities: softmax over the whole gallery (requirement 5)
    p = np.exp(sims / temperature); p /= p.sum()

    fig, axes = plt.subplots(1, k, figsize=(4 * k, 4.4))
    for ax, idx in zip(np.atleast_1d(axes), order):
        e = INDEX[idx]
        flow('fetch', f'load {e["path"].name} [tier={e["tier"]}]',
             'display + write description')
        ax.imshow(Image.open(e['path']).convert('RGB')); ax.axis('off')
        col = '#1a5276' if e['tier'] == 'SERVER' else '#0f766e'
        ax.set_title(f"{e['tier']}  |  P={p[idx]*100:.1f}%\n{e['name'][:38]}",
                     fontsize=8.5, color=col)
    plt.suptitle(f'Top-{k} for "{query}"   |   mobile {t_local:.2f}ms  '
                 f'vs  server {t_server:.0f}ms (same index, network is the '
                 f'difference)', fontsize=10.5)
    plt.tight_layout(); plt.show()

query = input('Search for one of the server photos, or your uploaded '
              'mobile photo: ')
search_and_show(query)

In [ ]:
# 6. (Optional) run more searches — try one query aimed at a SERVER photo
# and one aimed at your MOBILE photo, and watch the tier labels + timing.
search_and_show(input('Another query: '))

## What this demo proves, in one sentence
A photo indexed by a 30MB phone encoder and a photo indexed by a 1.5GB
server encoder sit in the **same index**, answer the **same free-text
queries**, and the only difference the user can measure is that the local
route skips the network.

**Honesty notes:** the network RTT is *simulated* (150ms, typical
cellular) since both "tiers" run in one Colab VM; probabilities are a
softmax over the current gallery (temperature 0.02), i.e. relative
confidence among these photos, not calibrated absolute probabilities; and
BLIP (the captioner) is a third model used for description only — it
plays no role in indexing or search.